In [34]:
import os
import cv2
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

In [35]:
IMG_HEIGHT = 64
IMG_WIDTH = 64

CELL_SIZE = 8
BLOCK_SIZE = 2
NBINS = 9

LBP_GRID_Y = 4
LBP_GRID_X = 4

In [36]:
roberts_x = np.array([[1,0],
                      [0,-1]], dtype=np.float32)

roberts_y = np.array([[0,1],
                      [-1,0]], dtype=np.float32)

In [37]:
def roberts_gradient(image):

    img = image.astype(np.float32)

    gx = cv2.filter2D(img, -1, roberts_x)
    gy = cv2.filter2D(img, -1, roberts_y)

    magnitude = np.sqrt(gx**2 + gy**2)
    angle = np.degrees(np.arctan2(gy, gx)) % 180

    return magnitude, angle

In [38]:
def compute_cell_histogram(magnitude, angle):

    hist = np.zeros(NBINS)

    bin_width = 180 / NBINS

    for i in range(magnitude.shape[0]):
        for j in range(magnitude.shape[1]):

            mag = magnitude[i,j]
            ang = angle[i,j]

            bin_index = int(ang // bin_width)
            if bin_index >= NBINS:
                bin_index = NBINS-1

            hist[bin_index] += mag

    return hist

In [39]:
def hog_feature(image):

    magnitude, angle = roberts_gradient(image)

    h, w = image.shape

    cells_y = h // CELL_SIZE
    cells_x = w // CELL_SIZE

    cell_hist = []

    for cy in range(cells_y):
        row = []
        for cx in range(cells_x):

            y = cy * CELL_SIZE
            x = cx * CELL_SIZE

            mag_patch = magnitude[y:y+CELL_SIZE, x:x+CELL_SIZE]
            ang_patch = angle[y:y+CELL_SIZE, x:x+CELL_SIZE]

            hist = compute_cell_histogram(mag_patch, ang_patch)

            row.append(hist)

        cell_hist.append(row)

    cell_hist = np.array(cell_hist)

    features = []

    for y in range(cells_y - BLOCK_SIZE + 1):
        for x in range(cells_x - BLOCK_SIZE + 1):

            block = cell_hist[y:y+BLOCK_SIZE, x:x+BLOCK_SIZE].flatten()

            norm = np.linalg.norm(block) + 1e-6
            block = block / norm

            features.extend(block)

    return np.array(features)

In [40]:
def compute_lbp(image):

    h, w = image.shape
    lbp = np.zeros((h,w), dtype=np.uint8)

    for i in range(1,h-1):
        for j in range(1,w-1):

            center = image[i,j]

            code = 0

            code |= (image[i-1,j-1] >= center) << 7
            code |= (image[i-1,j]   >= center) << 6
            code |= (image[i-1,j+1] >= center) << 5
            code |= (image[i,j+1]   >= center) << 4
            code |= (image[i+1,j+1] >= center) << 3
            code |= (image[i+1,j]   >= center) << 2
            code |= (image[i+1,j-1] >= center) << 1
            code |= (image[i,j-1]   >= center) << 0

            lbp[i,j] = code

    return lbp

In [41]:
def lbp_histogram_block(block):

    hist = np.zeros(256)

    for value in block.flatten():
        hist[value] += 1

    if hist.sum() > 0:
        hist = hist / hist.sum()

    return hist

In [42]:
def lbp_global_feature(image):

    lbp = compute_lbp(image)

    h, w = image.shape

    block_h = h // LBP_GRID_Y
    block_w = w // LBP_GRID_X

    features = []

    for y in range(LBP_GRID_Y):
        for x in range(LBP_GRID_X):

            yy = y * block_h
            xx = x * block_w

            block = lbp[yy:yy+block_h, xx:xx+block_w]

            hist = lbp_histogram_block(block)

            features.extend(hist)

    return np.array(features)

In [43]:
def extract_features(image):

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    gray = cv2.resize(gray, (IMG_WIDTH, IMG_HEIGHT))

    hog = hog_feature(gray)

    lbp = lbp_global_feature(gray)

    feature = np.concatenate((hog, lbp))

    return feature

In [44]:
def load_dataset(folder_path, label):

    features = []
    labels = []

    for file in os.listdir(folder_path):

        path = os.path.join(folder_path, file)

        img = cv2.imread(path)

        if img is None:
            continue

        feat = extract_features(img)

        features.append(feat)
        labels.append(label)

    return features, labels

In [45]:
base = "/kaggle/input/datasets/tongpython/cat-and-dog"

train_cats = base + "/training_set/training_set/cats"
train_dogs = base + "/training_set/training_set/dogs"

X_cat, y_cat = load_dataset(train_cats, 0)
X_dog, y_dog = load_dataset(train_dogs, 1)

X_train = np.array(X_cat + X_dog)
y_train = np.array(y_cat + y_dog)
print("Processing:")
print("Training samples:", X_train.shape)

Processing:
Training samples: (8005, 5860)


In [46]:
test_cats = base + "/test_set/test_set/cats"
test_dogs = base + "/test_set/test_set/dogs"

X_cat_test, y_cat_test = load_dataset(test_cats, 0)
X_dog_test, y_dog_test = load_dataset(test_dogs, 1)

X_test = np.array(X_cat_test + X_dog_test)
y_test = np.array(y_cat_test + y_dog_test)

print("Processing:")
print("Test samples:", X_test.shape)

Processing:
Test samples: (2023, 5860)


In [51]:
from sklearn.svm import SVC

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

svm = SVC(kernel='rbf', C=10, gamma='scale')

svm.fit(X_train, y_train)

SVC(C=10)

In [52]:
y_pred = svm.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7993079584775087

Classification Report

              precision    recall  f1-score   support

           0       0.80      0.80      0.80      1011
           1       0.80      0.80      0.80      1012

    accuracy                           0.80      2023
   macro avg       0.80      0.80      0.80      2023
weighted avg       0.80      0.80      0.80      2023


Confusion Matrix

[[811 200]
 [206 806]]


In [50]:
joblib.dump(svm,"hog_lbp_svm_model.pkl")

['hog_lbp_svm_model.pkl']